# S2 · Fundamentos de PySpark con OpenSky

AeroStreamPeru · Equipo 04 · Ciclo 9

Usaremos un día del histórico Bronze para practicar lectura con esquema explícito, transformaciones, agregaciones, evaluación perezosa y RDD. Calcularemos un primer resumen descriptivo de las dos dimensiones del equipo.

In [ ]:
from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    LongType,
    StringType,
    StructType,
)

bronze = Path("/opt/s01-arquitectura/data/2026")
bloque = bronze / "01" / "2026-01-01_2026-01-05.parquet"
dia_utc = date(2026, 1, 4)
inicio_utc = datetime.combine(dia_utc, time.min, tzinfo=timezone.utc)
fin_utc = inicio_utc + timedelta(days=1)

spark = (
    SparkSession.builder.master("local[2]")
    .appName("AeroStreamPeru-S02")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Periodo UTC: [{inicio_utc.isoformat()}, {fin_utc.isoformat()})")

## 1. Lectura y transformaciones

El esquema declara las siete columnas de Bronze que usamos. El filtro temporal se aplica sobre los segundos Unix de origen y luego se agrega una columna de fecha y hora para agrupar por ventana.

In [ ]:
esquema = (
    StructType()
    .add("time", LongType())
    .add("icao24", StringType())
    .add("lat", DoubleType())
    .add("lon", DoubleType())
    .add("onground", BooleanType())
    .add("velocity", DoubleType())
    .add("callsign", StringType())
)
estados = spark.read.schema(esquema).parquet(str(bloque))

estados_del_dia = (
    estados.filter(
        (F.col("time") >= int(inicio_utc.timestamp()))
        & (F.col("time") < int(fin_utc.timestamp()))
    )
    .withColumn("captura_utc", F.timestamp_seconds("time"))
    .withColumn("icao24", F.lower(F.trim("icao24")))
    .withColumn("callsign", F.trim("callsign"))
)

estados_del_dia.printSchema()
estados_del_dia.explain(True)

## 2. Del plan a la ejecución

En el plan físico, revisa `PushedFilters`. El lector Parquet recibe los límites de `time` y puede descartar bloques incompatibles con el filtro. `filter` y `withColumn` construyen el plan, y `explain` lo muestra. La acción `show` ejecuta trabajo para leer las primeras filas necesarias.

In [ ]:
estados_del_dia.select("captura_utc", "icao24", "onground", "velocity", "callsign").show(5)
print(f"Estados en el día de muestra: {estados_del_dia.count():,}")

## 3. Agrupaciones por captura y ventana

Para el volumen, contamos aeronaves distintas por captura y promediamos esos conteos en ventanas de diez minutos. Una captura que contiene estados, pero ninguna aeronave elegible, aporta cero. Un segundo sin registros no aparece como captura observada.

Para la velocidad, calculamos la mediana aproximada sobre las observaciones elegibles con mediciones entre 10 y 350 m/s, excluyendo los extremos. Este rango es exploratorio. En S3 se validarán las reglas definitivas de calidad, frescura y cobertura.

In [ ]:
identificador_valido = F.col("icao24").rlike(r"^[0-9a-f]{6}$")
posicion_en_zona = F.col("lat").between(-18.35, -0.04) & F.col("lon").between(-81.33, -68.65)
aeronave_en_vuelo = identificador_valido & posicion_en_zona & (~F.col("onground"))
velocidad_valida = (F.col("velocity") > 10) & (F.col("velocity") < 350)

volumen_por_captura = estados_del_dia.groupBy("captura_utc").agg(
    F.countDistinct(F.when(aeronave_en_vuelo, F.col("icao24"))).alias("aeronaves_en_vuelo")
)
volumen_por_ventana = volumen_por_captura.groupBy(
    F.window("captura_utc", "10 minutes").alias("ventana")
).agg(
    F.count("*").alias("capturas_observadas"),
    F.round(F.avg("aeronaves_en_vuelo"), 2).alias("volumen_promedio"),
)
en_vuelo = estados_del_dia.filter(aeronave_en_vuelo)

velocidad_por_ventana = (
    en_vuelo.filter(velocidad_valida)
    .groupBy(F.window("captura_utc", "10 minutes").alias("ventana"))
    .agg(
        F.round(F.percentile_approx("velocity", 0.5), 2).alias("velocidad_mediana_m_s"),
        F.count("*").alias("mediciones_de_velocidad"),
    )
)
resumen = (
    volumen_por_ventana.join(velocidad_por_ventana, "ventana", "left")
    .select(
        F.col("ventana.start").alias("inicio_utc"),
        "capturas_observadas",
        "volumen_promedio",
        "velocidad_mediana_m_s",
        "mediciones_de_velocidad",
    )
    .orderBy("inicio_utc")
)
resumen.show(8, truncate=False)

## 4. Ejercicio RDD

Usamos `map` y `reduceByKey` para contar estados por aeronave en una hora. `takeOrdered` ejecuta las operaciones y devuelve solo los cinco resultados solicitados. Un estado es una observación de la aeronave. Este conteo ayuda a ver por qué el volumen se calcula con identificadores distintos por captura.

In [ ]:
primera_hora_en_vuelo = en_vuelo.filter(
    F.col("time") < int((inicio_utc + timedelta(hours=1)).timestamp())
)
estados_por_aeronave = (
    primera_hora_en_vuelo.select("icao24").rdd
    .map(lambda fila: (fila.icao24, 1))
    .reduceByKey(lambda anterior, actual: anterior + actual)
)
for icao24, observaciones in estados_por_aeronave.takeOrdered(
    5, key=lambda par: (-par[1], par[0])
):
    print(f"{icao24}: {observaciones} estados")

## 5. Hallazgo y reflexión

Una aeronave puede generar muchos estados dentro de una hora. Si se contaran filas en la métrica de Leonardo, una aeronave con más observaciones aumentaría el total varias veces. La agrupación por captura permite medir cuántas se observan simultáneamente. Para Joel, mostramos cuántas mediciones respaldan cada mediana. En S3 comprobaremos cómo influyen la cobertura, los nulos y los duplicados.

Para la evidencia individual, ejecuta el notebook y explica el resultado de tu dimensión y cómo Spark aplica el filtro de tiempo en el plan. La guía pide que las capturas muestren el reloj y el usuario o perfil.

In [ ]:
spark.stop()